# Online Softmax — Solution

Reference implementation for `01_online_softmax_exercise.ipynb`.

## Setup

In [1]:
import torch

## Solution 1 — `online_softmax`

In [5]:
print(list(range(0,20,5)))

[0, 5, 10, 15]


In [59]:
def online_softmax(x, chunk_size):
    running_sum = 0
    m = float("-inf") 

    for i in range(0,len(x),chunk_size):
        chunk = x[i:i+chunk_size]
        chunk_max = torch.max(chunk)

        m_new = max(m,chunk_max)
        correction_factor = torch.exp(m-m_new)
        
        running_sum = running_sum*correction_factor + sum(torch.exp(chunk-m_new))
        m = m_new

    return torch.exp(x-m)/running_sum

In [60]:
online_softmax(torch.arange(20),5)

tensor([3.5416e-09, 9.6272e-09, 2.6169e-08, 7.1136e-08, 1.9337e-07, 5.2563e-07,
        1.4288e-06, 3.8839e-06, 1.0557e-05, 2.8698e-05, 7.8010e-05, 2.1205e-04,
        5.7642e-04, 1.5669e-03, 4.2592e-03, 1.1578e-02, 3.1471e-02, 8.5548e-02,
        2.3254e-01, 6.3212e-01])

**Why is this exact, not approximate?** Every chunk's contribution is correctly weighted in the final sum because the correction factor `exp(m_old - m_new)` retroactively normalizes prior contributions to the new global max. The mathematical identity `e^{x_j - m_new} = e^{x_j - m_old} · e^{m_old - m_new}` makes the correction perfectly equivalent — no information is lost._

## Solution 2 — Verify across many inputs

In [68]:
torch.manual_seed(0)

# Sweep across distributions, sequence lengths, and chunk sizes.
configs = [
    ('uniform[0,1)',     torch.rand(100)),
    ('standard normal',  torch.randn(100)),
    ('large positive',   torch.randn(100) * 10 + 50),
    ('large negative',   torch.randn(100) * 10 - 50),
    ('extreme values',   torch.tensor([-1000.0, -500.0, 0.0, 500.0, 1000.0] * 20)),
    ('length 7 (prime)', torch.randn(7)),
    ('length 1',         torch.randn(1)),
]

for name, x in configs:
    expected = torch.softmax(x, dim=-1)
    for chunk_size in [1, 3, 16, len(x)]:
        if chunk_size > len(x):
            continue
        got = online_softmax(x, chunk_size)
        max_err = (got - expected).abs().max().item()
        assert torch.allclose(got, expected, atol=1e-5), \
            f'{name}, chunk_size={chunk_size}: max abs err = {max_err:.2e}'
        print(f'  {name:<20}  chunk={chunk_size:>3}  max abs err = {max_err:.2e}')

print('\nOnline softmax matches torch.softmax across all distributions and chunk sizes.')

  uniform[0,1)          chunk=  1  max abs err = 3.73e-09
  uniform[0,1)          chunk=  3  max abs err = 3.73e-09
  uniform[0,1)          chunk= 16  max abs err = 2.79e-09
  uniform[0,1)          chunk=100  max abs err = 1.86e-09
  standard normal       chunk=  1  max abs err = 7.45e-09
  standard normal       chunk=  3  max abs err = 1.12e-08
  standard normal       chunk= 16  max abs err = 3.73e-09
  standard normal       chunk=100  max abs err = 7.45e-09
  large positive        chunk=  1  max abs err = 5.96e-08
  large positive        chunk=  3  max abs err = 5.96e-08
  large positive        chunk= 16  max abs err = 9.31e-10
  large positive        chunk=100  max abs err = 5.96e-08
  large negative        chunk=  1  max abs err = 1.19e-07
  large negative        chunk=  3  max abs err = 1.19e-07
  large negative        chunk= 16  max abs err = 1.19e-07
  large negative        chunk=100  max abs err = 1.19e-07
  extreme values        chunk=  1  max abs err = 0.00e+00
  extreme valu

## Run the tests

In [69]:
from tests import run_online_softmax_tests
run_online_softmax_tests(online_softmax)

Running online_softmax...
  ✓ Matches blog's example [2,8,1,9,3,7]
  ✓ chunk_size=1 (every-element)
  ✓ chunk_size=full row (no chunking)
  ✓ Random lengths × chunk sizes
  ✓ Numerically stable on extreme values

  All 5 checks passed ✓



True